In [1]:
!pip install easyocr opencv-python-headless Pillow numpy tqdm -q
print('\n✅ All libraries installed successfully!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 118.3 MB/s eta 0:00:00

✅ All libraries installed successfully!


In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('\n✅ Google Drive mounted successfully!')
print('Your Drive is now accessible at: /content/drive/MyDrive/')

Mounted at /content/drive

✅ Google Drive mounted successfully!
Your Drive is now accessible at: /content/drive/MyDrive/


In [21]:
import os

# ⬇️ CHANGE THIS to your actual folder path
INPUT_FOLDER = '/content/drive/MyDrive/Dataset/Internship/receipt_ocr/data/receipts'

# Output folder (will be created automatically)
OUTPUT_FOLDER = '/content/outputs'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Check how many images are found
SUPPORTED = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}
if os.path.exists(INPUT_FOLDER):
    images = [f for f in os.listdir(INPUT_FOLDER) if os.path.splitext(f)[1].lower() in SUPPORTED]
    print(f'✅ Found {len(images)} receipt image(s) in {INPUT_FOLDER}')
    for img in images:
        print(f'   📄 {img}')
else:
    print(f'❌ Folder not found: {INPUT_FOLDER}')
    print('Please create the folder in your Drive and add receipt images to it.')


✅ Found 371 receipt image(s) in /content/drive/MyDrive/Dataset/Internship/receipt_ocr/data/receipts
   📄 X51005442327.jpg
   📄 X51005568900.jpg
   📄 X51005303661.jpg
   📄 X51005741944.jpg
   📄 X51005605295.jpg
   📄 X51005447841.jpg
   📄 X51005230648.jpg
   📄 X51005441402.jpg
   📄 X51005719874.jpg
   📄 X51005719856.jpg
   📄 X51005663317.jpg
   📄 19.jpg
   📄 X51005433541.jpg
   📄 X51005749912.jpg
   📄 X51005719863.jpg
   📄 X51005711452.jpg
   📄 X51005719873.jpg
   📄 X51005757304.jpg
   📄 X51005442382.jpg
   📄 X51005442346.jpg
   📄 X51005763999.jpg
   📄 X51005745213.jpg
   📄 X51005361898.jpg
   📄 X51005745298.jpg
   📄 X51005447860.jpg
   📄 X51005757286.jpg
   📄 X51005676538.jpg
   📄 X51005806695.jpg
   📄 X51005444046.jpg
   📄 X51005577192.jpg
   📄 X51005757292.jpg
   📄 X51005677329.jpg
   📄 X51005677335.jpg
   📄 X51005715456.jpg
   📄 X51005711403.jpg
   📄 X51005442341.jpg
   📄 X51005663310.jpg
   📄 X51005757324.jpg
   📄 X51005676544.jpg
   📄 X51005442366.jpg
   📄 X51005746207.jpg
   📄 X51

In [22]:
import cv2
import numpy as np
from PIL import Image
import logging
logging.basicConfig(level=logging.WARNING)

def preprocess_image(image_path):
    """Full preprocessing: resize → deskew → contrast → denoise → binarize"""
    try:
        img = cv2.imread(image_path)
        if img is None:
            pil_img = Image.open(image_path).convert('RGB')
            img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

        # 1. Resize if too small
        h, w = img.shape[:2]
        if w < 800:
            scale = 800 / w
            img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

        # 2. Correct skew
        try:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 50, 150, apertureSize=3)
            lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=100, maxLineGap=10)
            if lines is not None:
                angles = []
                for line in lines:
                    x1,y1,x2,y2 = line[0]
                    if x2 != x1:
                        angle = np.degrees(np.arctan2(y2-y1, x2-x1))
                        if -45 < angle < 45:
                            angles.append(angle)
                if angles:
                    median_angle = np.median(angles)
                    if abs(median_angle) > 0.5:
                        h2, w2 = img.shape[:2]
                        M = cv2.getRotationMatrix2D((w2//2, h2//2), median_angle, 1.0)
                        img = cv2.warpAffine(img, M, (w2,h2), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        except:
            pass

        # 3. Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape)==3 else img

        # 4. CLAHE contrast enhancement
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        gray = clahe.apply(gray)

        # 5. Denoise
        gray = cv2.fastNlMeansDenoising(gray, h=10)

        # 6. Binarize (adaptive threshold)
        result = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                        cv2.THRESH_BINARY, 11, 2)
        return result

    except Exception as e:
        print(f'  ⚠ Preprocessing warning for {image_path}: {e}')
        pil = Image.open(image_path).convert('L')
        return np.array(pil)

print('✅ Preprocessing functions loaded.')

✅ Preprocessing functions loaded.


In [8]:
import easyocr

print('Loading EasyOCR model... (this takes 1-2 minutes first time)')
reader = easyocr.Reader(['en'], gpu=False, verbose=False)
print('✅ EasyOCR ready!')

def run_ocr(image):
    """Run OCR and return list of (text, confidence) tuples."""
    raw = reader.readtext(image, detail=1, paragraph=False)
    results = [(text.strip(), float(conf)) for (_, text, conf) in raw if text.strip()]
    return results

def average_confidence(results):
    if not results:
        return 0.0
    return sum(c for _, c in results) / len(results)

print('✅ OCR functions loaded.')

Loading EasyOCR model... (this takes 1-2 minutes first time)


✅ EasyOCR ready!
✅ OCR functions loaded.


In [9]:
import re
from typing import List, Dict, Optional

DATE_PATTERNS = [
    r'\b(\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4})\b',
    r'\b(\d{4}[\/\-\.]\d{1,2}[\/\-\.]\d{1,2})\b',
    r'\b(\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{2,4})\b',
    r'\b((?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4})\b',
]

TOTAL_KEYWORDS = [
    r'\bTOTAL\b', r'\bGRAND\s+TOTAL\b', r'\bAMOUNT\s+DUE\b',
    r'\bBALANCE\s+DUE\b', r'\bNET\s+TOTAL\b', r'\bFINAL\s+TOTAL\b',
    r'\bTOTAL\s+AMOUNT\b', r'\bAMT\s+DUE\b',
]

PRICE_RE = re.compile(r'(?:Rs\.?|INR|\u20b9|\$|\u20ac|\u00a3)?\s*(\d{1,6}(?:[,\.]\d{2,3})*(?:\.\d{2})?)')

STORE_NOISE = {'receipt','invoice','bill','tax','gst','vat','thank','you',
               'visit','again','cashier','customer','copy','tel','phone','www','http'}

SKIP_ITEM_RE = re.compile(
    r'\b(total|subtotal|sub-total|tax|gst|vat|service|discount|change|cash|card|tip|balance|due)\b', re.I)

def combine_confidence(ocr_conf, text_conf, w=0.3):
    return round(min(max(w * ocr_conf + (1-w) * text_conf, 0.0), 1.0), 3)

def clean_price(raw):
    if not raw: return None
    raw = re.sub(r'[^\d.,]', '', raw).strip()
    if not raw: return None
    if re.match(r'^\d{1,3}(,\d{3})+(\. \d{2})?$', raw):
        raw = raw.replace(',', '')
    return raw

def extract_store_name(lines, ocr_conf):
    candidates = []
    for i, line in enumerate(lines[:8]):
        if len(line) < 3 or len(line) > 60: continue
        if re.search(r'\d{5,}', line): continue
        words = line.lower().split()
        if any(w in STORE_NOISE for w in words): continue
        if PRICE_RE.search(line) and any(c.isdigit() for c in line[:5]): continue
        score = 1.0 - i * 0.08
        if line.isupper(): score += 0.15
        if line.istitle(): score += 0.10
        digit_ratio = sum(c.isdigit() for c in line) / max(len(line), 1)
        score -= digit_ratio * 0.5
        score = min(max(score, 0.1), 1.0)
        candidates.append((line, score))
    if not candidates:
        return {'value': None, 'confidence': 0.0, 'flagged': True}
    best, text_conf = sorted(candidates, key=lambda x: -x[1])[0]
    conf = combine_confidence(ocr_conf, text_conf, w=0.4)
    result = {'value': best, 'confidence': conf}
    if conf < 0.7: result['flagged'] = True
    return result

def extract_date(lines, ocr_conf):
    for line in lines:
        for pat in DATE_PATTERNS:
            m = re.search(pat, line, re.IGNORECASE)
            if m:
                labelled = bool(re.search(r'\b(date|dated|dt)\b', line, re.I))
                text_conf = 0.95 if labelled else 0.80
                conf = combine_confidence(ocr_conf, text_conf)
                result = {'value': m.group(1), 'confidence': conf}
                if conf < 0.7: result['flagged'] = True
                return result
    return {'value': None, 'confidence': 0.0, 'flagged': True}

def extract_total(lines, ocr_conf):
    candidates = []
    for line in lines:
        if any(re.search(kw, line.upper()) for kw in TOTAL_KEYWORDS):
            m = PRICE_RE.search(line)
            if m:
                amount = clean_price(m.group(1))
                if amount:
                    conf = combine_confidence(ocr_conf, 0.95)
                    candidates.append((amount, conf))
    if candidates:
        amount, conf = candidates[-1]  # last = grand total
        result = {'value': amount, 'confidence': conf}
        if conf < 0.7: result['flagged'] = True
        return result
    # Fallback: largest number
    all_prices = []
    for line in lines:
        for m in PRICE_RE.finditer(line):
            val = clean_price(m.group(1))
            if val:
                try: all_prices.append((float(val.replace(',','')), val))
                except: pass
    if all_prices:
        _, largest = sorted(all_prices, reverse=True)[0]
        conf = combine_confidence(ocr_conf, 0.45)
        return {'value': largest, 'confidence': conf, 'flagged': True}
    return {'value': None, 'confidence': 0.0, 'flagged': True}

def extract_items(lines, ocr_conf):
    items = []
    for line in lines:
        if SKIP_ITEM_RE.search(line): continue
        m = re.match(r'^(.{2,40}?)\s{2,}(?:\d+\s*[xX×]\s*)?(?:Rs\.?|INR|\u20b9|\$)?\s*(\d{1,6}(?:[,\.]\d{2,3})*(?:\.\d{2})?)\s*$', line)
        if m:
            name = m.group(1).strip()
            price = clean_price(m.group(2))
            if name and price and not any(re.search(p, name, re.I) for p in DATE_PATTERNS):
                items.append({'name': name, 'price': price,
                              'confidence': combine_confidence(ocr_conf, 0.80)})
    return items

def extract_all(ocr_results, ocr_conf):
    lines = [text for text, _ in ocr_results]
    flags = []

    store = extract_store_name(lines, ocr_conf)
    date  = extract_date(lines, ocr_conf)
    total = extract_total(lines, ocr_conf)
    items = extract_items(lines, ocr_conf)

    for fname, field in [('store_name', store), ('date', date), ('total_amount', total)]:
        if field.get('flagged'):
            flags.append(f'LOW_CONFIDENCE:{fname}({field["confidence"]:.2f})')
    if not items:
        flags.append('NO_ITEMS_DETECTED')

    return {
        'store_name': store,
        'date': date,
        'items': items,
        'total_amount': total,
        'ocr_confidence': round(ocr_conf, 3),
        'flags': flags
    }

print('✅ Extractor functions loaded.')

✅ Extractor functions loaded.


In [ ]:
import json
from pathlib import Path
import cv2
import numpy as np

# ============================================================
# IMPROVED PREPROCESSING FUNCTION (replaces old one)
# ============================================================
def preprocess_image(image_path):
    try:
        img = cv2.imread(image_path)
        if img is None:
            from PIL import Image as PILImage
            pil_img = PILImage.open(image_path).convert('RGB')
            img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

        # Bigger resize = better OCR accuracy
        h, w = img.shape[:2]
        if w < 1200:
            scale = 1200 / w
            img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape)==3 else img

        # Stronger contrast enhancement
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        gray = clahe.apply(gray)

        # Sharpen text edges
        kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
        gray = cv2.filter2D(gray, -1, kernel)

        # Denoise
        gray = cv2.fastNlMeansDenoising(gray, h=8)

        # Binarize
        result = cv2.adaptiveThreshold(gray, 255,
                    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                    cv2.THRESH_BINARY, 15, 4)
        return result

    except Exception as e:
        from PIL import Image as PILImage
        pil = PILImage.open(image_path).convert('L')
        return np.array(pil)


# ============================================================
# IMPROVED DATE PATTERNS (catches more formats)
# ============================================================
DATE_PATTERNS = [
    r'\b(\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4})\b',
    r'\b(\d{4}[\/\-\.]\d{1,2}[\/\-\.]\d{1,2})\b',
    r'\b(\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{2,4})\b',
    r'\b((?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4})\b',
    r'\b(\d{1,2}\s+\d{1,2}\s+\d{2,4})\b',
    r'\b(20\d{2}[01]\d[0-3]\d)\b',
    r'(\d{1,2}[\/\-]\d{1,2}[\/\-]\d{2,4})',
    r'\b(20\d{2})\b',
]


# ============================================================
# IMPROVED TOTAL KEYWORDS (covers parking, restaurants, etc.)
# ============================================================
TOTAL_KEYWORDS = [
    r'\bTOTAL\b', r'\bGRAND\s+TOTAL\b', r'\bAMOUNT\s+DUE\b',
    r'\bBALANCE\s+DUE\b', r'\bNET\s+TOTAL\b', r'\bFINAL\s+TOTAL\b',
    r'\bTOTAL\s+AMOUNT\b', r'\bAMT\s+DUE\b', r'\bAMOUNT\b',
    r'\bCHARGE\b', r'\bFEE\b', r'\bPAYABLE\b',
    r'\bPAID\b', r'\bNET\b', r'\bSUM\b',
    r'\bDUE\b', r'\bBILL\b', r'\bINVOICE\s+TOTAL\b',
]

PRICE_RE = re.compile(r'(?:Rs\.?|INR|\u20b9|\$|\u20ac|\u00a3)?\s*(\d{1,6}(?:[,\.]\d{2,3})*(?:\.\d{2})?)')
SKIP_ITEM_RE = re.compile(
    r'\b(total|subtotal|sub-total|tax|gst|vat|service|discount|change|cash|card|tip|balance|due)\b', re.I)
STORE_NOISE = {'receipt','invoice','bill','tax','gst','vat','thank','you',
               'visit','again','cashier','customer','copy','tel','phone','www','http'}


# ============================================================
# MULTI-PREPROCESSING: tries 3 versions, picks best
# ============================================================
def preprocess_best(image_path):
    img = cv2.imread(image_path)
    if img is None:
        from PIL import Image as PILImage
        img = cv2.cvtColor(np.array(PILImage.open(image_path).convert('RGB')), cv2.COLOR_RGB2BGR)

    h, w = img.shape[:2]
    if w < 1200:
        img = cv2.resize(img, None, fx=1200/w, fy=1200/w, interpolation=cv2.INTER_CUBIC)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape)==3 else img
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))

    versions = {
        'adaptive': cv2.adaptiveThreshold(
            cv2.fastNlMeansDenoising(clahe.apply(gray), h=8),
            255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 15, 4),

        'otsu': cv2.threshold(
            cv2.GaussianBlur(gray,(3,3),0),
            0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)[1],

        'sharpened': cv2.filter2D(
            cv2.fastNlMeansDenoising(clahe.apply(gray), h=8),
            -1, np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])),
    }

    best_img, best_conf = None, -1
    best_name = ''
    for name, processed in versions.items():
        results = run_ocr(processed)
        conf = average_confidence(results)
        if conf > best_conf:
            best_conf = conf
            best_img = processed
            best_name = name

    print(f'  ✔ Best preprocessing: [{best_name}] confidence={best_conf:.3f}')
    return best_img


# ============================================================
# IMPROVED EXTRACTION FUNCTIONS
# ============================================================
def combine_confidence(ocr_conf, text_conf, w=0.3):
    return round(min(max(w * ocr_conf + (1-w) * text_conf, 0.0), 1.0), 3)

def clean_price(raw):
    if not raw: return None
    raw = re.sub(r'[^\d.,]', '', raw).strip()
    if not raw: return None
    if re.match(r'^\d{1,3}(,\d{3})+(\.\d{2})?$', raw):
        raw = raw.replace(',', '')
    return raw

def extract_store_name(lines, ocr_conf):
    candidates = []
    for i, line in enumerate(lines[:8]):
        if len(line) < 3 or len(line) > 60: continue
        if re.search(r'\d{5,}', line): continue
        words = line.lower().split()
        if any(w in STORE_NOISE for w in words): continue
        score = 1.0 - i * 0.08
        if line.isupper(): score += 0.15
        if line.istitle(): score += 0.10
        digit_ratio = sum(c.isdigit() for c in line) / max(len(line), 1)
        score -= digit_ratio * 0.5
        score = min(max(score, 0.1), 1.0)
        candidates.append((line, score))
    if not candidates:
        return {'value': None, 'confidence': 0.0, 'flagged': True}
    best, text_conf = sorted(candidates, key=lambda x: -x[1])[0]
    conf = combine_confidence(ocr_conf, text_conf, w=0.4)
    result = {'value': best, 'confidence': conf}
    if conf < 0.7: result['flagged'] = True
    return result

def extract_date(lines, ocr_conf):
    for line in lines:
        for pat in DATE_PATTERNS:
            m = re.search(pat, line, re.IGNORECASE)
            if m:
                labelled = bool(re.search(r'\b(date|dated|dt|time)\b', line, re.I))
                text_conf = 0.95 if labelled else 0.82
                conf = combine_confidence(ocr_conf, text_conf)
                result = {'value': m.group(1), 'confidence': conf}
                if conf < 0.7: result['flagged'] = True
                return result
    return {'value': None, 'confidence': 0.0, 'flagged': True}

def extract_total(lines, ocr_conf):
    candidates = []
    for line in lines:
        if any(re.search(kw, line.upper()) for kw in TOTAL_KEYWORDS):
            m = PRICE_RE.search(line)
            if m:
                amount = clean_price(m.group(1))
                if amount:
                    try:
                        val = float(amount.replace(',',''))
                        if val > 0:  # ignore zeros
                            conf = combine_confidence(ocr_conf, 0.95)
                            candidates.append((val, amount, conf))
                    except: pass
    if candidates:
        # pick largest among keyword matches (most likely grand total)
        _, amount, conf = sorted(candidates, reverse=True)[0]
        result = {'value': amount, 'confidence': conf}
        if conf < 0.7: result['flagged'] = True
        return result
    # Fallback: largest number on page
    all_prices = []
    for line in lines:
        for m in PRICE_RE.finditer(line):
            val = clean_price(m.group(1))
            if val:
                try: all_prices.append((float(val.replace(',','')), val))
                except: pass
    if all_prices:
        _, largest = sorted(all_prices, reverse=True)[0]
        conf = combine_confidence(ocr_conf, 0.45)
        return {'value': largest, 'confidence': conf, 'flagged': True}
    return {'value': None, 'confidence': 0.0, 'flagged': True}

def extract_items(lines, ocr_conf):
    items = []
    for line in lines:
        if SKIP_ITEM_RE.search(line): continue
        m = re.match(
            r'^(.{2,40}?)\s{2,}(?:\d+\s*[xX×]\s*)?(?:Rs\.?|INR|₹|\$)?\s*(\d{1,6}(?:[,\.]\d{2,3})*(?:\.\d{2})?)\s*$',
            line)
        if m:
            name = m.group(1).strip()
            price = clean_price(m.group(2))
            if name and price:
                items.append({
                    'name': name,
                    'price': price,
                    'confidence': combine_confidence(ocr_conf, 0.80)
                })
    return items

def extract_all(ocr_results, ocr_conf):
    lines = [text for text, _ in ocr_results]
    flags = []
    store = extract_store_name(lines, ocr_conf)
    date  = extract_date(lines, ocr_conf)
    total = extract_total(lines, ocr_conf)
    items = extract_items(lines, ocr_conf)
    for fname, field in [('store_name',store),('date',date),('total_amount',total)]:
        if field.get('flagged'):
            flags.append(f'LOW_CONFIDENCE:{fname}({field["confidence"]:.2f})')
    if not items:
        flags.append('NO_ITEMS_DETECTED')
    return {
        'store_name': store,
        'date': date,
        'items': items,
        'total_amount': total,
        'ocr_confidence': round(ocr_conf, 3),
        'flags': flags
    }


# ============================================================
# MAIN PIPELINE — processes all images
# ============================================================
SUPPORTED = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}

input_path = Path(INPUT_FOLDER)
output_path = Path(OUTPUT_FOLDER)
output_path.mkdir(parents=True, exist_ok=True)

image_files = sorted([f for f in input_path.iterdir()
                       if f.suffix.lower() in SUPPORTED])

if not image_files:
    print('❌ No images found! Check INPUT_FOLDER in Step 3.')
else:
    print(f'🔄 Processing {len(image_files)} image(s)...\n')
    all_results = []

    for i, img_file in enumerate(image_files):
        print(f'[{i+1}/{len(image_files)}] {img_file.name}')
        result = {'filename': img_file.name, 'error': None}

        try:
            # IMPROVED: tries 3 preprocessings, picks best
            preprocessed = preprocess_best(str(img_file))

            # OCR
            ocr_results = run_ocr(preprocessed)
            ocr_conf = average_confidence(ocr_results)
            print(f'  ✔ OCR: {len(ocr_results)} words | avg conf: {ocr_conf:.3f}')

            if not ocr_results:
                result.update({'error': 'No text detected',
                               'flags': ['NO_TEXT_DETECTED'],
                               'ocr_confidence': 0.0})
            else:
                extracted = extract_all(ocr_results, ocr_conf)
                result.update(extracted)
                print(f'  ✔ Store : {result["store_name"]["value"]}')
                print(f'  ✔ Date  : {result["date"]["value"]}')
                print(f'  ✔ Total : {result["total_amount"]["value"]}')
                print(f'  ✔ Items : {len(result["items"])} found')
                if result['flags']:
                    print(f'  ⚠ Flags : {result["flags"]}')

        except Exception as e:
            result['error'] = str(e)
            print(f'  ❌ Error: {e}')

        # Save individual JSON
        out_file = output_path / f'{img_file.stem}.json'
        with open(out_file, 'w') as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        print(f'  💾 Saved → {out_file.name}\n')

        all_results.append(result)

    # Save combined
    with open(output_path / 'all_receipts.json', 'w') as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    print(f'✅ All done! {len(image_files)} receipts processed.')
    print(f'📁 Outputs in: {OUTPUT_FOLDER}')

🔄 Processing 371 image(s)...

[1/371] 0.jpg
  ✔ Best preprocessing: [otsu] confidence=0.667
  ✔ OCR: 43 words | avg conf: 0.667
  ✔ Store : WAL-MART
  ✔ Date  : 2003
  ✔ Total : 010451
  ✔ Items : 0 found
  ⚠ Flags : ['LOW_CONFIDENCE:total_amount(0.52)', 'NO_ITEMS_DETECTED']
  💾 Saved → 0.json

[2/371] 1.jpg
  ✔ Best preprocessing: [otsu] confidence=0.672
  ✔ OCR: 74 words | avg conf: 0.672
  ✔ Store : TRADER JOE'S
  ✔ Date  : 2001
  ✔ Total : 75206
  ✔ Items : 0 found
  ⚠ Flags : ['LOW_CONFIDENCE:total_amount(0.52)', 'NO_ITEMS_DETECTED']
  💾 Saved → 1.json

[3/371] 10.jpg
  ✔ Best preprocessing: [sharpened] confidence=0.684
  ✔ OCR: 88 words | avg conf: 0.684
  ✔ Store : SFAR
  ✔ Date  : None
  ✔ Total : 448124
  ✔ Items : 0 found
  ⚠ Flags : ['LOW_CONFIDENCE:date(0.00)', 'LOW_CONFIDENCE:total_amount(0.52)', 'NO_ITEMS_DETECTED']
  💾 Saved → 10.json

[4/371] 11.jpg


In [11]:
from collections import defaultdict

def parse_amount(value):
    if not value: return 0.0
    try:
        return float(str(value).replace(',','').replace('₹','').replace('$','').strip())
    except:
        return 0.0

total_spend = 0.0
num_tx = 0
num_tx_with_total = 0
store_spend = defaultdict(float)
store_count = defaultdict(int)
low_conf_receipts = []
flagged_count = 0

for receipt in all_results:
    fname = receipt.get('filename', 'unknown')
    store = receipt.get('store_name', {}).get('value') or 'Unknown Store'
    total_field = receipt.get('total_amount', {})
    total_val = total_field.get('value')
    total_conf = total_field.get('confidence', 0.0)
    ocr_conf = receipt.get('ocr_confidence', 0.0)

    num_tx += 1
    store_count[store] += 1

    if total_val and total_conf >= 0.5:
        amount = parse_amount(total_val)
        if amount > 0:
            total_spend += amount
            store_spend[store] += amount
            num_tx_with_total += 1

    if ocr_conf < 0.6:
        low_conf_receipts.append(fname)
    for f in receipt.get('flags', []):
        if 'LOW_CONFIDENCE' in f:
            flagged_count += 1

spend_per_store = {
    store: {
        'total_spend': round(amt, 2),
        'transactions': store_count[store],
        'average_per_transaction': round(amt / store_count[store], 2)
    }
    for store, amt in sorted(store_spend.items(), key=lambda x: -x[1])
}

summary = {
    'summary': {
        'total_spend': round(total_spend, 2),
        'number_of_transactions': num_tx,
        'transactions_with_detected_total': num_tx_with_total,
        'average_transaction_value': round(total_spend / num_tx_with_total, 2) if num_tx_with_total > 0 else 0.0
    },
    'spend_per_store': spend_per_store,
    'reliability': {
        'low_confidence_receipt_images': low_conf_receipts,
        'total_flagged_fields': flagged_count,
        'note': 'Transactions with total_amount confidence < 0.5 excluded from totals'
    }
}

# Save summary
summary_path = output_path / 'expense_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

# Pretty print
s = summary['summary']
print('=' * 50)
print('       💰 EXPENSE SUMMARY REPORT')
print('=' * 50)
print(f'  Total Spend              : {s["total_spend"]:.2f}')
print(f'  Number of Transactions   : {s["number_of_transactions"]}')
print(f'  Transactions with Total  : {s["transactions_with_detected_total"]}')
print(f'  Avg. Transaction Value   : {s["average_transaction_value"]:.2f}')
print()
print('  Spend per Store:')
for store, data in summary['spend_per_store'].items():
    print(f'    {store:<30} {data["total_spend"]:>10.2f}  ({data["transactions"]} tx)')
r = summary['reliability']
if r['low_confidence_receipt_images']:
    print(f'\n  ⚠  Low-confidence images  : {", ".join(r["low_confidence_receipt_images"])}')
if r['total_flagged_fields'] > 0:
    print(f'  ⚠  Total flagged fields   : {r["total_flagged_fields"]}')
print('=' * 50)
print(f'\n💾 Summary saved to: {summary_path}')

       💰 EXPENSE SUMMARY REPORT
  Total Spend              : 219343051.00
  Number of Transactions   : 371
  Transactions with Total  : 168
  Avg. Transaction Value   : 1305613.40

  Spend per Store:
    SECURE PARKING                 130088169.00  (1 tx)
    SYARIKAT PERNIAGAAN GIN KEE    18569760.00  (28 tx)
    KEDAI PAPAN YEW CHUAN          6140020.00  (7 tx)
    RESTORAN WAN SHENG             3937600.00  (5 tx)
    RESTORAN WAN  SHENG            2496138.00  (4 tx)
    YONG CEN ENTERPRISE            1963648.00  (2 tx)
    THREE                          1772918.00  (2 tx)
    POPULAR                         992000.00  (3 tx)
    POPULAR  Book                   992000.00  (1 tx)
    Vil                             992000.00  (1 tx)
    YONC CEN ENTERPRISE             981824.00  (1 tx)
    BEYOND BROTHERS HARDWARE        974280.00  (2 tx)
    Geoventure                      965857.00  (1 tx)
    BILLION SIX ENTERPRISE          956640.00  (1 tx)
    PERNIAGAAN ZHENG KUI            9565

In [13]:
import shutil
from google.colab import files

# Create a zip of all outputs
zip_path = '/content/receipt_ocr_outputs'
shutil.make_archive(zip_path, 'zip', OUTPUT_FOLDER)

print('📦 Zipping outputs...')
files.download(zip_path + '.zip')
print('\n✅ Download started! Check your browser downloads folder.')
print('\nFiles included:')
for f in sorted(output_path.iterdir()):
    print(f'  📄 {f.name}')

📦 Zipping outputs...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download started! Check your browser downloads folder.

Files included:
  📄 0.json
  📄 1.json
  📄 10.json
  📄 11.json
  📄 12.json
  📄 13.json
  📄 14.json
  📄 15.json
  📄 16.json
  📄 17.json
  📄 18.json
  📄 19.json
  📄 2.json
  📄 20.json
  📄 21.json
  📄 3.json
  📄 4.json
  📄 5.json
  📄 6.json
  📄 7.json
  📄 8.json
  📄 9.json
  📄 X00016469612.json
  📄 X00016469619.json
  📄 X00016469620.json
  📄 X00016469622.json
  📄 X00016469623.json
  📄 X00016469669.json
  📄 X00016469670.json
  📄 X00016469671.json
  📄 X00016469672.json
  📄 X00016469676.json
  📄 X51005200931.json
  📄 X51005200938.json
  📄 X51005230605.json
  📄 X51005230616.json
  📄 X51005230617.json
  📄 X51005230621.json
  📄 X51005230648.json
  📄 X51005230657.json
  📄 X51005230659.json
  📄 X51005255805.json
  📄 X51005268200.json
  📄 X51005268262.json
  📄 X51005268275.json
  📄 X51005268400.json
  📄 X51005268408.json
  📄 X51005268472.json
  📄 X51005288570.json
  📄 X51005301659.json
  📄 X51005301661.json
  📄 X51005301666.json
  📄 X510053

In [15]:
# Change this to any receipt filename (without .jpg/.png)
PREVIEW_FILE = 'X51005288570'   # ← change this

preview_path = output_path / f'{PREVIEW_FILE}.json'
if preview_path.exists():
    with open(preview_path) as f:
        data = json.load(f)
    print(json.dumps(data, indent=2))
else:
    print(f'File not found: {preview_path}')
    print('Available files:')
    for f in output_path.glob('*.json'):
        print(f'  {f.stem}')

{
  "filename": "X51005288570.jpg",
  "error": null,
  "store_name": {
    "value": "SECURE PARKING",
    "confidence": 0.885
  },
  "date": {
    "value": null,
    "confidence": 0.0,
    "flagged": true
  },
  "items": [],
  "total_amount": {
    "value": "130088,169",
    "confidence": 0.529,
    "flagged": true
  },
  "ocr_confidence": 0.713,
  "flags": [
    "LOW_CONFIDENCE:date(0.00)",
    "LOW_CONFIDENCE:total_amount(0.53)",
    "NO_ITEMS_DETECTED"
  ]
}
